# v2 results — suffix-automaton speculative decoding

Runs every experiment behind the write-up on a Colab T4:

1. **Correctness** — unit tests (exact sampling, greedy equivalence)
2. **Cost model** — layer counts, target verify cost vs. draft length k, draft/target cost ratio
3. **Repetitive prompt** — 5 modes × greedy & sampled × 5 repeats (mean ± std)
4. **Generic prompts** — 10 code prompts × 5 modes
5. Saves `results/v2_results.json` and figures in `results/figures/`

Runtime → Change runtime type → **T4 GPU**. Total ≈ 20–30 min.

In [ ]:
BRANCH = "main"
import os, sys
REPO = "hybrid-suffix-speculative-decoding"
if not os.path.isdir(REPO):
    !git clone -q -b {BRANCH} https://github.com/Matthew-Matta/{REPO}.git
os.chdir(REPO); sys.path.insert(0, os.getcwd())
!pip install -q transformers accelerate datasets pytest
!git log --oneline -3

## 1. Correctness tests (CPU, tiny random models — ~1 min)

In [ ]:
!python -m pytest -q tests

## 2. Load models

In [ ]:
import time, json, numpy as np, torch, matplotlib.pyplot as plt, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.speculative_decode import HybridSpecDecoder
from src.benchmark import load_dataset

DEVICE, DTYPE = "cuda", torch.float16
TARGET, DRAFT = "Qwen/Qwen2.5-Coder-1.5B-Instruct", "Qwen/Qwen2.5-Coder-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(TARGET)
target = AutoModelForCausalLM.from_pretrained(TARGET, torch_dtype=DTYPE).to(DEVICE).eval()
draft  = AutoModelForCausalLM.from_pretrained(DRAFT,  torch_dtype=DTYPE).to(DEVICE).eval()
dec = HybridSpecDecoder(target, tok, draft_model=draft, device=DEVICE)
dec.warmup()

def info(m):
    c = m.config
    return dict(params_B=round(sum(p.numel() for p in m.parameters())/1e9, 3), layers=c.num_hidden_layers,
                hidden=c.hidden_size, kv_heads=c.num_key_value_heads, head_dim=c.hidden_size//c.num_attention_heads)
model_info = {"target": info(target), "draft": info(draft), "gpu": torch.cuda.get_device_name()}
pd.DataFrame(model_info).T

## 3. Cost model — what does a forward pass actually cost?

* **Verify cost vs k**: time one target forward over `k` new tokens on top of a ~512-token KV cache.
  If decode were purely memory-bound and overhead-free, cost would be ~flat in k.
* **Draft vs target**: time a 1-token forward of each. Params ratio ≈ 0.33, but if we're
  launch/overhead-bound the time ratio tracks *layer count* instead.

In [ ]:
@torch.no_grad()
def time_forward(model, ctx_len, k, iters=30):
    ids = torch.randint(100, 1000, (1, ctx_len), device=DEVICE)
    kv = model(ids, use_cache=True).past_key_values
    new = torch.randint(100, 1000, (1, k), device=DEVICE)
    for _ in range(3):                                   # warm
        model(new, past_key_values=kv, use_cache=True); kv.crop(-k)
    torch.cuda.synchronize(); ts = []
    for _ in range(iters):
        t0 = time.perf_counter()
        model(new, past_key_values=kv, use_cache=True)
        torch.cuda.synchronize(); ts.append(time.perf_counter() - t0)
        kv.crop(-k)
    return float(np.median(ts)) * 1e3                   # ms

ks = [1, 2, 4, 6, 8, 10, 12, 16, 32, 64]
verify_ms = {k: time_forward(target, 512, k) for k in ks}
t_tgt, t_dft = verify_ms[1], time_forward(draft, 512, 1)
cost = {"target_1tok_ms": t_tgt, "draft_1tok_ms": t_dft, "draft_over_target": t_dft / t_tgt,
        "param_ratio": model_info["draft"]["params_B"] / model_info["target"]["params_B"],
        "layer_ratio": model_info["draft"]["layers"] / model_info["target"]["layers"],
        "verify_ms": verify_ms}
# Bandwidth roofline for a 1-token step (weights only)
weight_bytes = model_info["target"]["params_B"] * 1e9 * 2
cost["roofline_ms_T4_320GBps"] = weight_bytes / 320e9 * 1e3
print(json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in cost.items() if k != "verify_ms"}, indent=2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, [verify_ms[k] for k in ks], "o-", label="target forward (k new tokens)")
ax.axhline(t_dft, color="tab:orange", ls="--", label=f"draft 1-token forward ({t_dft:.1f} ms)")
ax.axhline(cost["roofline_ms_T4_320GBps"], color="gray", ls=":", label="bandwidth roofline (weights only)")
ax.set_xscale("log", base=2); ax.set_xlabel("tokens verified in one forward (k)"); ax.set_ylabel("ms")
ax.set_title("T4, Qwen2.5-Coder-1.5B, fp16, 512-token context"); ax.legend(); fig.tight_layout()
os.makedirs("results/figures", exist_ok=True); fig.savefig("results/figures/verify_cost_vs_k.png", dpi=150); plt.show()

## 4. Experiments

In [ ]:
SA_PROMPT = """Complete this Python Calculator class by implementing the remaining methods
following the same pattern as the examples below:

class Calculator:
    def add(self, x: float, y: float) -> float:
        \"\"\"Add x and y.\"\"\"
        return x + y

    def subtract(self, x: float, y: float) -> float:
        \"\"\"Subtract y from x.\"\"\"
        return x - y

    def multiply(self, x: float, y: float) -> float:
        # implement this

    def divide(self, x: float, y: float) -> float:
        # implement this, raise ValueError if y == 0

    def power(self, x: float, y: float) -> float:
        # implement this

    def modulo(self, x: float, y: float) -> float:
        # implement this"""

MODES = ["autoregressive", "specdec", "sa_only", "hybrid_fixed", "hybrid_dynamic"]
KW = dict(num_draft_tokens=4, sa_threshold=2, sa_max_draft_len=10)

def run(prompt, mode, temperature, max_new_tokens, seed):
    torch.manual_seed(seed)
    text, m = dec.generate(prompt, max_new_tokens=max_new_tokens, mode=mode, temperature=temperature, **KW)
    d = m.to_dict(); d.pop("draft_length_history"); d.pop("source_history")
    d["frac_steps"] = {s: m.source_history.count(s) / max(1, len(m.source_history)) for s in ("SA", "draft", "autoregressive")}
    return d, text

def summarize(rows):
    df = pd.DataFrame(rows)
    g = df.groupby("mode", sort=False)
    out = pd.DataFrame({
        "TPS": g.tokens_per_second.mean().round(1), "TPS_std": g.tokens_per_second.std().round(1),
        "tok/target_fwd": g.tokens_per_target_forward.mean().round(2),
        "SA_acc": g.sa_acceptance_rate.mean().round(3), "draft_acc": g.draft_acceptance_rate.mean().round(3),
        "avg_draft_len": g.avg_draft_length.mean().round(2), "n": g.size()})
    out["speedup"] = (out.TPS / out.loc["autoregressive", "TPS"]).round(2)
    return out

results = {"model_info": model_info, "cost": cost, "experiments": {}}

In [ ]:
# 4a. Repetitive prompt: greedy and sampled, 5 repeats each
REPEATS = 5
for temp in (0.0, 1.0):
    rows, texts = [], {}
    for mode in MODES:
        for r in range(REPEATS):
            d, text = run(SA_PROMPT, mode, temp, 300, seed=r)
            rows.append({"mode": mode, "repeat": r, **d}); texts.setdefault(mode, text)
    name = f"repetitive_temp{temp}"
    results["experiments"][name] = rows
    print(f"\n=== {name} ==="); display(summarize(rows))
    if temp == 0.0:   # greedy outputs should be identical across modes (fp16 may rarely diverge late)
        same = {m: texts[m] == texts["autoregressive"] for m in MODES}
        print("greedy output identical to AR:", same)

In [ ]:
# 4b. Generic prompts: 10 code prompts, greedy
prompts = load_dataset(n_samples=10)
rows = []
for mode in MODES:
    for i, p in enumerate(prompts):
        d, _ = run(p, mode, 0.0, 128, seed=i)
        rows.append({"mode": mode, "prompt": i, **d})
results["experiments"]["generic_temp0.0"] = rows
display(summarize(rows))

In [ ]:
# 5. Save + headline chart
with open("results/v2_results.json", "w") as f:
    json.dump(results, f, indent=2, default=float)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, name in zip(axes, ["repetitive_temp0.0", "repetitive_temp1.0", "generic_temp0.0"]):
    s = summarize(results["experiments"][name])
    ax.bar(s.index, s.TPS, yerr=s.TPS_std, capsize=4, color=plt.cm.tab10(range(len(s))))
    for i, (v, sp) in enumerate(zip(s.TPS, s.speedup)):
        ax.text(i, v + 1, f"{sp:.2f}x", ha="center", fontsize=9)
    ax.set_title(name); ax.tick_params(axis="x", rotation=25)
axes[0].set_ylabel("tokens / s (mean ± std)"); fig.tight_layout()
fig.savefig("results/figures/v2_tps.png", dpi=150); plt.show()
print("Saved results/v2_results.json — download it (Files panel) and share it back.")